In [61]:
from collections import OrderedDict
import itertools as it

In [28]:
type SubstrMap = dict[str, tuple[int, list[tuple[int, int]]]]

In [40]:
def get_substr(s: str) -> SubstrMap:
    """Get non-overlapping substrings from a string with its value count and inclusion positions"""
    pr = OrderedDict()
    all_subs = [] # should be an orderedset
    for l in range(1, len(s)):
        for start in range(0, len(s) - 1):
            # Get all possible substrings
            sub = s[start:start + l]
            if sub not in all_subs:
                all_subs.append(sub)

    for sub in all_subs:
        # scan for the positions:
        start = 0
        while start < (len(s) - len(sub) + 1):
            if s[start:start + len(sub)] == sub:
                if pr.get(sub) is not None:
                    old = pr[sub]
                    pr[sub] = (old[0] + 1, old[1] + [(start, start + len(sub))])
                else:
                    pr[sub] = (1, [(start, start + len(sub))])
                start += len(sub) # skip over
            else:
                start += 1
    return pr


In [78]:
def mut_strong(pr: SubstrMap) -> tuple[dict[str, list[str]], dict[str, list[str]]]:
    """Find strongly mutually exclusive substrings, returns the s2->{s1} and s1->{s2} mappings and the {s1, s2} pairs"""
    mut = OrderedDict()  # cannot contain: s2 -> s1
    cbi = OrderedDict()  # cannot be in: s1 -> s2
    pairs = []
    for i, (s1, (count1, pos1)) in enumerate(pr.items()):
        for j, (s2, (count2, pos2)) in enumerate(pr.items()):
            
            # strong condition: # of inclusions must be the same
            if j == i or (not s1 in s2) or len(pos1) == 0 or len(pos1) != len(pos2):
                continue
            # also skip subs of size 1
            #if count1 == 1 or count2 == 1:
            #    continue
            
            # Check strong condition
            # We assume that sls is sorted
            for k in range(len(pos2)):
                if not (pos1[k][0] >= pos2[k][0] and pos1[k][1] <= pos2[k][1]):
                    continue
            pairs.append((s1, s2))
            if mut.get(s2) is not None:
                mut[s2].append(s1)
            else:
                mut[s2] = [s1]
            if cbi.get(s1) is not None:
                cbi[s1].append(s2)
            else:
                cbi[s1] = [s2]
    return mut, cbi, pairs


In [3]:
s = "1111[11[1[0]0]1[0]0]11[1[0]0]1[0]0"

In [41]:
pr = get_substr(s)

In [42]:
pr

OrderedDict([('1',
              (12,
               [(0, 1),
                (1, 2),
                (2, 3),
                (3, 4),
                (5, 6),
                (6, 7),
                (8, 9),
                (14, 15),
                (20, 21),
                (21, 22),
                (23, 24),
                (29, 30)])),
             ('[',
              (7,
               [(4, 5),
                (7, 8),
                (9, 10),
                (15, 16),
                (22, 23),
                (24, 25),
                (30, 31)])),
             ('0',
              (8,
               [(10, 11),
                (12, 13),
                (16, 17),
                (18, 19),
                (25, 26),
                (27, 28),
                (31, 32),
                (33, 34)])),
             (']',
              (7,
               [(11, 12),
                (13, 14),
                (17, 18),
                (19, 20),
                (26, 27),
                (28, 29),
   

## Filter out

In [44]:
len(pr)  # All substrings

441

In [45]:
len([x for x in pr if len(x) > len(s) / 2])  # Substrings which are larger than half of the string

152

In [46]:
len([x for x in pr if len(x) == 1])  # Unit substrings

4

In [48]:
len([x for x in pr.values() if x[0] == 1])  # Substrings which occur only once
# Use this as a decent-enough heuristic for long enough recursion depths

358

In [19]:
pr

{'1': 12,
 '[': 7,
 '0': 8,
 ']': 7,
 '11': 4,
 '1[': 7,
 '[1': 3,
 '[0': 4,
 '0]': 7,
 ']0': 4,
 ']1': 3,
 '111': 1,
 '11[': 3,
 '1[1': 3,
 '[11': 1,
 '[1[': 2,
 '1[0': 4,
 '[0]': 4,
 '0]0': 4,
 ']0]': 3,
 '0]1': 3,
 ']1[': 2,
 ']11': 1,
 '1111': 1,
 '111[': 1,
 '11[1': 2,
 '1[11': 1,
 '[11[': 1,
 '1[1[': 2,
 '[1[0': 2,
 '1[0]': 4,
 '[0]0': 4,
 '0]0]': 3,
 ']0]1': 3,
 '0]1[': 2,
 ']1[0': 2,
 '0]11': 1,
 ']11[': 1,
 '1111[': 1,
 '111[1': 1,
 '11[11': 1,
 '1[11[': 1,
 '[11[1': 1,
 '11[1[': 2,
 '1[1[0': 2,
 '[1[0]': 2,
 '1[0]0': 4,
 '[0]0]': 3,
 '0]0]1': 3,
 ']0]1[': 2,
 '0]1[0': 2,
 ']1[0]': 2,
 ']0]11': 1,
 '0]11[': 1,
 ']11[1': 1,
 '1111[1': 1,
 '111[11': 1,
 '11[11[': 1,
 '1[11[1': 1,
 '[11[1[': 1,
 '11[1[0': 2,
 '1[1[0]': 2,
 '[1[0]0': 2,
 '1[0]0]': 3,
 '[0]0]1': 3,
 '0]0]1[': 2,
 ']0]1[0': 2,
 '0]1[0]': 2,
 ']1[0]0': 2,
 '0]0]11': 1,
 ']0]11[': 1,
 '0]11[1': 1,
 ']11[1[': 1,
 '1111[11': 1,
 '111[11[': 1,
 '11[11[1': 1,
 '1[11[1[': 1,
 '[11[1[0': 1,
 '11[1[0]': 2,
 '1[1[0]0': 2,
 '[

In [51]:
pr_f = OrderedDict([(k, v) for k, v in pr.items() if v[0] > 1 and len(k) > 1])
pr_f

OrderedDict([('11', (4, [(0, 2), (2, 4), (5, 7), (20, 22)])),
             ('1[',
              (7,
               [(3, 5),
                (6, 8),
                (8, 10),
                (14, 16),
                (21, 23),
                (23, 25),
                (29, 31)])),
             ('[1', (3, [(4, 6), (7, 9), (22, 24)])),
             ('[0', (4, [(9, 11), (15, 17), (24, 26), (30, 32)])),
             ('0]',
              (7,
               [(10, 12),
                (12, 14),
                (16, 18),
                (18, 20),
                (25, 27),
                (27, 29),
                (31, 33)])),
             (']0', (4, [(11, 13), (17, 19), (26, 28), (32, 34)])),
             (']1', (3, [(13, 15), (19, 21), (28, 30)])),
             ('11[', (3, [(2, 5), (5, 8), (20, 23)])),
             ('1[1', (3, [(3, 6), (6, 9), (21, 24)])),
             ('[1[', (2, [(7, 10), (22, 25)])),
             ('1[0', (4, [(8, 11), (14, 17), (23, 26), (29, 32)])),
             ('[0]', (4,

In [79]:
# Find mutually exclusive substrings
mut, cbi, pairs = mut_strong(pr_f)

In [81]:
len(pairs)

735

# Gene mapping algorithms

## Count (naive)

In [72]:
list(pr_f.items())[0]

('11', (4, [(0, 2), (2, 4), (5, 7), (20, 22)]))

In [76]:
groups = []

get_count = lambda x: x[1][0]
data = sorted(pr_f.items(), key=get_count)  # is stable
for key, group in it.groupby(data, get_count):
    groups.append((key, list(group)))

In [77]:
groups

[(2,
  [('[1[', (2, [(7, 10), (22, 25)])),
   (']1[', (2, [(13, 16), (28, 31)])),
   ('11[1', (2, [(2, 6), (20, 24)])),
   ('1[1[', (2, [(6, 10), (21, 25)])),
   ('[1[0', (2, [(7, 11), (22, 26)])),
   ('0]1[', (2, [(12, 16), (27, 31)])),
   (']1[0', (2, [(13, 17), (28, 32)])),
   ('11[1[', (2, [(5, 10), (20, 25)])),
   ('1[1[0', (2, [(6, 11), (21, 26)])),
   ('[1[0]', (2, [(7, 12), (22, 27)])),
   (']0]1[', (2, [(11, 16), (26, 31)])),
   ('0]1[0', (2, [(12, 17), (27, 32)])),
   (']1[0]', (2, [(13, 18), (28, 33)])),
   ('11[1[0', (2, [(5, 11), (20, 26)])),
   ('1[1[0]', (2, [(6, 12), (21, 27)])),
   ('[1[0]0', (2, [(7, 13), (22, 28)])),
   ('0]0]1[', (2, [(10, 16), (25, 31)])),
   (']0]1[0', (2, [(11, 17), (26, 32)])),
   ('0]1[0]', (2, [(12, 18), (27, 33)])),
   (']1[0]0', (2, [(13, 19), (28, 34)])),
   ('11[1[0]', (2, [(5, 12), (20, 27)])),
   ('1[1[0]0', (2, [(6, 13), (21, 28)])),
   ('[1[0]0]', (2, [(7, 14), (22, 29)])),
   ('1[0]0]1', (2, [(8, 15), (23, 30)])),
   ('[0]0]1[', (2, [

## Lex

In [84]:
pr_lex = OrderedDict(sorted(pr_f.items(), key=lambda x: x[0]))

In [85]:
groups_lex = []
cur_len = 0

for k, v in pr_lex.items():
    if cur_len != 0 and len(k) > cur_len:
        groups_lex[-1].append((k, v))
    else:
        # create new group
        groups_lex.append([(k, v)])
    cur_len = len(k)

groups_lex

[[('0]',
   (7,
    [(10, 12), (12, 14), (16, 18), (18, 20), (25, 27), (27, 29), (31, 33)])),
  ('0]0', (4, [(10, 13), (16, 19), (25, 28), (31, 34)])),
  ('0]0]', (3, [(10, 14), (16, 20), (25, 29)])),
  ('0]0]1', (3, [(10, 15), (16, 21), (25, 30)])),
  ('0]0]1[', (2, [(10, 16), (25, 31)])),
  ('0]0]1[0', (2, [(10, 17), (25, 32)])),
  ('0]0]1[0]', (2, [(10, 18), (25, 33)])),
  ('0]0]1[0]0', (2, [(10, 19), (25, 34)]))],
 [('0]1', (3, [(12, 15), (18, 21), (27, 30)])),
  ('0]1[', (2, [(12, 16), (27, 31)])),
  ('0]1[0', (2, [(12, 17), (27, 32)])),
  ('0]1[0]', (2, [(12, 18), (27, 33)])),
  ('0]1[0]0', (2, [(12, 19), (27, 34)]))],
 [('11', (4, [(0, 2), (2, 4), (5, 7), (20, 22)])),
  ('11[', (3, [(2, 5), (5, 8), (20, 23)])),
  ('11[1', (2, [(2, 6), (20, 24)])),
  ('11[1[', (2, [(5, 10), (20, 25)])),
  ('11[1[0', (2, [(5, 11), (20, 26)])),
  ('11[1[0]', (2, [(5, 12), (20, 27)])),
  ('11[1[0]0', (2, [(5, 13), (20, 28)])),
  ('11[1[0]0]', (2, [(5, 14), (20, 29)])),
  ('11[1[0]0]1', (2, [(5, 15),

## Seed

In [108]:
# Get all seeds
s1_seeds = OrderedDict()

next_seed = 0
for s1, s2_set in cbi.items():
    # check if the seed already exists for s1 and {s2}
    cur_seeds = set()
    if s1 in s1_seeds:
        cur_seeds.add(s1_seeds[s1])
    # We still have to check the seeds of {s2} in case if there are overlapping seed groups
    for s2 in s2_set:
        if s2 in s1_seeds:  # s2 may have a larger substring counterpart
            cur_seeds.add(s1_seeds[s2])

    if len(cur_seeds) == 0:
        # New seed
        s1_seeds[s1] = next_seed
        for s2 in s2_set:
            s1_seeds[s2] = next_seed
        next_seed += 1
    elif len(cur_seeds) == 1:
        # apply the existing seed
        new_seed = list(cur_seeds)[0]
        s1_seeds[s1] = new_seed
        for s2 in s2_set:
            s1_seeds[s2] = new_seed
    else:
        # seed groups intersect, need to merge
        print(f"seed groups intersect : {cur_seeds}")
        new_seed = list(cur_seeds)[0]
        # O(n) update
        for k in s1_seeds:
            if s1_seeds[k] in cur_seeds:
                s1_seeds[k] = new_seed
        
        s1_seeds[s1] = new_seed
        for s2 in s2_set:
            s1_seeds[s2] = new_seed

num_seeds = next_seed

In [98]:
s1_seeds

OrderedDict([('[1', 0),
             ('1[1', 0),
             ('[0', 1),
             ('1[0', 1),
             ('[0]', 1),
             ('1[0]', 1),
             ('[0]0', 1),
             ('1[0]0', 1),
             (']0', 1),
             ('0]0', 1),
             (']1', 2),
             ('0]1', 2),
             (']0]1', 2),
             ('0]0]1', 2),
             ('[0]0]1', 2),
             ('[1[', 3),
             ('1[1[', 3),
             ('[1[0', 3),
             ('11[1[', 3),
             ('1[1[0', 3),
             ('[1[0]', 3),
             ('11[1[0', 3),
             ('1[1[0]', 3),
             ('[1[0]0', 3),
             ('11[1[0]', 3),
             ('1[1[0]0', 3),
             ('[1[0]0]', 3),
             ('11[1[0]0', 3),
             ('1[1[0]0]', 3),
             ('[1[0]0]1', 3),
             ('11[1[0]0]', 3),
             ('1[1[0]0]1', 3),
             ('[1[0]0]1[', 3),
             ('11[1[0]0]1', 3),
             ('1[1[0]0]1[', 3),
             ('[1[0]0]1[0', 3),
           

In [105]:
[x for x in pr_f if x not in s1_seeds]  # substr without a seed

['11', '1[', '0]', '11[']

In [110]:
pr_seeds = OrderedDict()

next_seed = num_seeds
for k, v in pr_f.items():
    if k in s1_seeds:
        cur_seed = s1_seeds[k]
    else:
        cur_seed = next_seed
        next_seed += 1
        
    pr_seeds[k] = (v[0], cur_seed, v[1])

In [111]:
pr_seeds

OrderedDict([('11', (4, 4, [(0, 2), (2, 4), (5, 7), (20, 22)])),
             ('1[',
              (7,
               5,
               [(3, 5),
                (6, 8),
                (8, 10),
                (14, 16),
                (21, 23),
                (23, 25),
                (29, 31)])),
             ('[1', (3, 0, [(4, 6), (7, 9), (22, 24)])),
             ('[0', (4, 1, [(9, 11), (15, 17), (24, 26), (30, 32)])),
             ('0]',
              (7,
               6,
               [(10, 12),
                (12, 14),
                (16, 18),
                (18, 20),
                (25, 27),
                (27, 29),
                (31, 33)])),
             (']0', (4, 1, [(11, 13), (17, 19), (26, 28), (32, 34)])),
             (']1', (3, 2, [(13, 15), (19, 21), (28, 30)])),
             ('11[', (3, 7, [(2, 5), (5, 8), (20, 23)])),
             ('1[1', (3, 0, [(3, 6), (6, 9), (21, 24)])),
             ('[1[', (2, 3, [(7, 10), (22, 25)])),
             ('1[0', (4, 1, [(8,

In [114]:
# Group by seed
groups_seed = []

get_seed = lambda x: x[1][1]
data = sorted(pr_seeds.items(), key=get_seed)  # is stable
for key, group in it.groupby(data, get_seed):
    # Sort by lex in the seed group
    group_lex = sorted(map(lambda y: (y[0], (y[1][0], y[1][2])), group), key=lambda x: x[0])
    groups_seed.append((key, list(group_lex)))

In [115]:
groups_seed

[(0,
  [('1[1', (3, [(3, 6), (6, 9), (21, 24)])),
   ('[1', (3, [(4, 6), (7, 9), (22, 24)]))]),
 (1,
  [('0]0', (4, [(10, 13), (16, 19), (25, 28), (31, 34)])),
   ('1[0', (4, [(8, 11), (14, 17), (23, 26), (29, 32)])),
   ('1[0]', (4, [(8, 12), (14, 18), (23, 27), (29, 33)])),
   ('1[0]0', (4, [(8, 13), (14, 19), (23, 28), (29, 34)])),
   ('[0', (4, [(9, 11), (15, 17), (24, 26), (30, 32)])),
   ('[0]', (4, [(9, 12), (15, 18), (24, 27), (30, 33)])),
   ('[0]0', (4, [(9, 13), (15, 19), (24, 28), (30, 34)])),
   (']0', (4, [(11, 13), (17, 19), (26, 28), (32, 34)]))]),
 (2,
  [('0]0]', (3, [(10, 14), (16, 20), (25, 29)])),
   ('0]0]1', (3, [(10, 15), (16, 21), (25, 30)])),
   ('0]1', (3, [(12, 15), (18, 21), (27, 30)])),
   ('1[0]0]', (3, [(8, 14), (14, 20), (23, 29)])),
   ('[0]0]', (3, [(9, 14), (15, 20), (24, 29)])),
   ('[0]0]1', (3, [(9, 15), (15, 21), (24, 30)])),
   (']0]', (3, [(11, 14), (17, 20), (26, 29)])),
   (']0]1', (3, [(11, 15), (17, 21), (26, 30)])),
   (']1', (3, [(13, 15)